In [28]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import OneHotEncoder

train = pd.read_csv('csv/train.csv')
test = pd.read_csv('csv/test.csv')
sample_submission = pd.read_csv('csv/sample_submission.csv')

#특성과 타겟 변수 분리
train = train.drop(columns=['ID'], axis = 1)
test = test.drop(columns=['ID'], axis = 1)


In [29]:
train[['국가']].shape

(4376, 1)

In [30]:
# 설립연도 -> 2025-설립연도
train['설립연도'] =2025- train['설립연도']
test['설립연도'] =2025- test['설립연도']

category_features = ['국가','분야','투자단계','기업가치(백억원)']
numeric_features = ['설립연도','직원 수','고객수(백만명)','총 투자금(억원)','연매출(억원)','SNS 팔로워 수(백만명)']
bool_features = ['인수여부','상장여부']

# OneHotEncoder 객체를 각 범주형 feature별로 따로 저장하여 사용
encoders = {}

# 범주형 데이터를 encoding
for feature in category_features:
    encoders[feature] = OneHotEncoder(sparse_output=False)
    train[feature] = train[feature].fillna('Missing')
    test[feature] = test[feature].fillna('Missing')
    encoded = encoders[feature].fit_transform(train[[feature]])
    encoded_df = pd.DataFrame(encoded, columns=encoders[feature].get_feature_names_out([feature]))
    train=pd.concat([train.drop(columns=feature).reset_index(drop=True),encoded_df],axis=1)
    
    encoded = encoders[feature].transform(test[[feature]])
    encoded_df = pd.DataFrame(encoded, columns=encoders[feature].get_feature_names_out([feature]))
    test=pd.concat([test.drop(columns=feature).reset_index(drop=True),encoded_df],axis=1)

# 불리언 값을 0과 1로 변환 ('Yes' → 1, 'No' → 0 으로 변환)
bool_map = {'Yes': 1, 'No': 0}

for feature in bool_features:
    train[feature] = train[feature].map(bool_map)
    test[feature] = test[feature].map(bool_map)

# 수치형 변수 결측치를 평균값으로 대체
for feature in numeric_features:
    mean_value = train[feature].mean()
    train[feature] = train[feature].fillna(mean_value)
    test[feature] = test[feature].fillna(mean_value)


In [33]:
print(train)

      설립연도    직원 수  인수여부  상장여부   고객수(백만명)  총 투자금(억원)  연매출(억원)  SNS 팔로워 수(백만명)  \
0       16  4126.0     0     0  56.000000     3365.0   4764.0            4.71   
1        2  4167.0     1     0  80.000000     4069.0    279.0            1.00   
2        7  3132.0     1     1  54.000000     6453.0  12141.0            4.00   
3        9  3245.0     1     1  49.214332      665.0  10547.0            2.97   
4        5  1969.0     0     1  94.000000      829.0   9810.0            1.00   
...    ...     ...   ...   ...        ...        ...      ...             ...   
4371     4  4841.0     1     0  90.000000     4187.0   9394.0            4.00   
4372     5   555.0     0     1  37.000000      796.0   2969.0            3.00   
4373     2   506.0     0     1  49.214332     3314.0   4512.0            1.47   
4374    24  1438.0     0     0  53.000000     2395.0   3755.0            5.00   
4375     8  3499.0     0     1  41.000000      903.0   9417.0            5.00   

      성공확률  국가_CT001  ...  

In [35]:
from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

# 예시 데이터
X_train, X_test, y_train, y_test = train_test_split(train.drop(columns=['성공확률']), train['성공확률'], test_size=0.2, random_state=42)

# 모델 생성
model = XGBRegressor()

# 학습
model.fit(X_train, y_train)

# 예측
y_pred = model.predict(X_test)

# 평가
mse = mean_squared_error(y_test, y_pred)
print(f'MSE: {mse:.4f}')

MSE: 0.0677


In [36]:
from sklearn.model_selection import GridSearchCV

model = XGBRegressor()

param_grid = {
    'n_estimators': [100, 300, 500],
    'learning_rate': [0.01, 0.05, 0.1],
    'max_depth': [3, 5],
    'subsample': [0.6, 0.8, 1.0],
}

grid_search = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    cv=5,  # 5-fold 교차검증
    scoring='neg_mean_squared_error',  # 회귀라서 MSE 기준
    n_jobs=-1,  # 가능한 모든 CPU 사용
    verbose=1
)

# 학습
grid_search.fit(X_train, y_train)

# 최적의 파라미터 조합
print("Best parameters:", grid_search.best_params_)
print("Best score (MSE):", -grid_search.best_score_)

Fitting 5 folds for each of 54 candidates, totalling 270 fits
Best parameters: {'learning_rate': 0.01, 'max_depth': 5, 'n_estimators': 100, 'subsample': 0.6}
Best score (MSE): 0.05824595867933867


In [38]:
# 모델 파라미터 조정
model = XGBRegressor(
    n_estimators=100,
    learning_rate=0.01,
    max_depth=5,
    subsample=0.6
)
model.fit(X_train, y_train)

# 테스트 데이터에 대한 확률 예측
y_pred = model.predict(X_test)

#점수
mse = mean_squared_error(y_test, y_pred)
print(mse)

0.058263616410721225


In [39]:
#제출
model = XGBRegressor(
    n_estimators=100,
    learning_rate=0.01,
    max_depth=5,
    subsample=0.6
)
model.fit(train.drop(columns=['성공확률']), train['성공확률'])

pred = model.predict(test)

sample_submission['성공확률'] = pred
sample_submission.to_csv('./baseline_submission.csv', index = False, encoding = 'utf-8-sig')

In [41]:
#RandomForest

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV

model = RandomForestRegressor()

param_grid = {
    'n_estimators': [100, 200, 300],         # 트리 수
    'max_depth': [None, 10, 20, 30],         # 트리 최대 깊이
    'min_samples_split': [2, 5, 10],         # 노드 분할 최소 샘플 수
    'min_samples_leaf': [1, 2, 4],           # 리프 노드 최소 샘플 수
    'max_features': ['sqrt', 'log2', None],  # 특성 샘플링 방법
    'bootstrap': [True, False]               # 배깅 여부
}

grid_search = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    cv=5,  # 5-fold 교차검증
    scoring='neg_mean_squared_error',  # 회귀라서 MSE 기준
    n_jobs=-1,  # 가능한 모든 CPU 사용
    verbose=1
)

# 학습
grid_search.fit(X_train, y_train)

# 최적의 파라미터 조합
print("Best parameters:", grid_search.best_params_)
print("Best score (MSE):", -grid_search.best_score_)

Fitting 5 folds for each of 648 candidates, totalling 3240 fits


KeyboardInterrupt: 

In [57]:
# 모델 파라미터 조정
model = RandomForestRegressor(
    n_estimators=100,

    max_depth=10

)
model.fit(X_train, y_train)

# 테스트 데이터에 대한 확률 예측
y_pred = model.predict(X_test)

#점수
mse = mean_squared_error(y_test, y_pred)
print(mse)

0.058375562690422184


In [58]:
#제출
model = RandomForestRegressor(
    n_estimators=100,

    max_depth=10

)
model.fit(train.drop(columns=['성공확률']), train['성공확률'])

pred = model.predict(test)

sample_submission['성공확률'] = pred
sample_submission.to_csv('./baseline_submission.csv', index = False, encoding = 'utf-8-sig')